# RDFSolve Paper Analysis

This notebook produces statistics and figures for the paper:
**"Systematic Analysis of Connectivity in the Life Sciences Linked Open Data Cloud"**

## Contents

1. Data Loading & Overview
2. Schema Statistics
3. Vocabulary Distribution Analysis
4. Cross-Dataset Connectivity
5. Instance-Level Mappings
6. Paper Figures Generation

In [ ]:
# Standard imports
from __future__ import annotations

import json
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

# RDFSolve imports
from rdfsolve import VERSION
from rdfsolve.overlap import jaccard_similarity

warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"RDFSolve version: {VERSION}")

In [ ]:
# Configuration
OUTPUT_DIR = Path("../../output")  # Relative to notebooks/
FIGURES_DIR = Path("./figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Check output directory exists
if not OUTPUT_DIR.exists():
    # Try absolute path
    OUTPUT_DIR = Path("/home/javier.millanacosta/rdfsolve/output")

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Figures directory: {FIGURES_DIR.resolve()}")

## 1. Data Loading & Overview

In [ ]:
def load_schemas(output_dir: Path) -> dict:
    """Load all schema files from output directory."""
    schemas = {}
    
    for pattern in ["*_mined_remote_schema.jsonld", "*_discovered_remote_schema.jsonld"]:
        for f in output_dir.glob(pattern):
            name = f.stem.replace("_mined_remote_schema", "").replace("_discovered_remote_schema", "")
            try:
                data = json.loads(f.read_text())
                
                # Skip empty schemas
                if not data.get("@graph"):
                    continue
                    
                # Extract classes and properties
                classes = set()
                properties = set()
                namespaces = set()
                patterns = []
                
                for item in data.get("@graph", []):
                    class_uri = item.get("@id", "")
                    if class_uri:
                        classes.add(class_uri)
                        # Extract namespace
                        if "#" in class_uri:
                            namespaces.add(class_uri.rsplit("#", 1)[0] + "#")
                        elif "/" in class_uri:
                            namespaces.add(class_uri.rsplit("/", 1)[0] + "/")
                    
                    for p in item.get("patterns", []):
                        prop = p.get("property", "")
                        if prop:
                            properties.add(prop)
                        patterns.append(p)
                
                schemas[name] = {
                    "raw": data,
                    "classes": classes,
                    "properties": properties,
                    "namespaces": namespaces,
                    "patterns": patterns,
                    "about": data.get("@about", {}),
                    "file": f,
                }
            except Exception as e:
                print(f"Failed to load {f.name}: {e}")
    
    return schemas


def load_mappings(output_dir: Path) -> dict:
    """Load mapping files."""
    mappings = {
        "sssom": [],
        "semra": [],
        "instance_matching": [],
        "class_derived": [],
        "inferenced": [],
    }
    
    mappings_dir = output_dir / "mappings"
    if not mappings_dir.exists():
        return mappings
    
    for subdir, key in [
        ("sssom", "sssom"),
        ("semra", "semra"),
        ("instance_matching", "instance_matching"),
        ("class_derived", "class_derived"),
        ("inferenced", "inferenced"),
    ]:
        dir_path = mappings_dir / subdir
        if dir_path.exists():
            for f in dir_path.glob("*.jsonld"):
                try:
                    data = json.loads(f.read_text())
                    mappings[key].append({
                        "file": f.name,
                        "data": data,
                        "edge_count": len(data.get("@graph", [])),
                    })
                except Exception as e:
                    print(f"Failed to load {f}: {e}")
    
    return mappings


# Load data
print("Loading schemas...")
schemas = load_schemas(OUTPUT_DIR)
print(f"Loaded {len(schemas)} schemas")

print("\nLoading mappings...")
mappings = load_mappings(OUTPUT_DIR)
for k, v in mappings.items():
    if v:
        print(f"  {k}: {len(v)} files")

## 2. Schema Statistics

In [ ]:
# Compute schema statistics
schema_stats = []

for name, schema in schemas.items():
    schema_stats.append({
        "name": name,
        "classes": len(schema["classes"]),
        "properties": len(schema["properties"]),
        "patterns": len(schema["patterns"]),
        "namespaces": len(schema["namespaces"]),
    })

df_schemas = pd.DataFrame(schema_stats).sort_values("classes", ascending=False)

# Summary statistics
print("=" * 60)
print("SCHEMA STATISTICS SUMMARY")
print("=" * 60)
print(f"Total schemas: {len(df_schemas)}")
print(f"Total classes: {df_schemas['classes'].sum():,}")
print(f"Total properties: {df_schemas['properties'].sum():,}")
print(f"Total patterns: {df_schemas['patterns'].sum():,}")
print()
print("Distribution:")
print(df_schemas[["classes", "properties", "patterns"]].describe())

In [ ]:
# Top schemas by size
print("\nTop 20 schemas by class count:")
display(df_schemas.head(20))

In [ ]:
# Figure: Schema size distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Classes distribution
axes[0].hist(df_schemas["classes"], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel("Number of Classes")
axes[0].set_ylabel("Count")
axes[0].set_title("Class Count Distribution")
axes[0].set_yscale('log')

# Properties distribution
axes[1].hist(df_schemas["properties"], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel("Number of Properties")
axes[1].set_ylabel("Count")
axes[1].set_title("Property Count Distribution")
axes[1].set_yscale('log')

# Patterns distribution
axes[2].hist(df_schemas["patterns"], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[2].set_xlabel("Number of Patterns")
axes[2].set_ylabel("Count")
axes[2].set_title("Pattern Count Distribution")
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig(FIGURES_DIR / "schema_size_distribution.png", dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / "schema_size_distribution.pdf", bbox_inches='tight')
plt.show()

## 3. Vocabulary Distribution Analysis

In [ ]:
# Collect all namespaces with their usage counts
namespace_usage = Counter()
namespace_to_datasets = defaultdict(set)

for name, schema in schemas.items():
    for ns in schema["namespaces"]:
        namespace_usage[ns] += 1
        namespace_to_datasets[ns].add(name)

# Known namespace prefixes
KNOWN_PREFIXES = {
    "http://www.w3.org/1999/02/22-rdf-syntax-ns#": "rdf",
    "http://www.w3.org/2000/01/rdf-schema#": "rdfs",
    "http://www.w3.org/2002/07/owl#": "owl",
    "http://www.w3.org/2004/02/skos/core#": "skos",
    "http://xmlns.com/foaf/0.1/": "foaf",
    "http://purl.org/dc/terms/": "dcterms",
    "http://purl.org/dc/elements/1.1/": "dc",
    "http://schema.org/": "schema",
    "http://purl.obolibrary.org/obo/": "obo",
    "http://www.w3.org/ns/prov#": "prov",
    "http://rdfs.org/ns/void#": "void",
    "http://www.w3.org/2001/XMLSchema#": "xsd",
    "http://semanticscience.org/resource/": "sio",
    "http://purl.org/ontology/bibo/": "bibo",
}

def get_prefix(ns: str) -> str:
    """Get short prefix for namespace."""
    if ns in KNOWN_PREFIXES:
        return KNOWN_PREFIXES[ns]
    # Extract from URL
    parts = ns.rstrip("/#").split("/")
    return parts[-1] if parts else ns[:20]

# Top namespaces
print("=" * 60)
print("TOP 30 NAMESPACES BY USAGE")
print("=" * 60)

ns_data = []
for ns, count in namespace_usage.most_common(30):
    ns_data.append({
        "namespace": ns,
        "prefix": get_prefix(ns),
        "dataset_count": count,
        "percent": count / len(schemas) * 100,
    })

df_ns = pd.DataFrame(ns_data)
display(df_ns)

In [ ]:
# Figure: Top namespaces bar chart
fig, ax = plt.subplots(figsize=(12, 8))

top_ns = df_ns.head(20)
colors = ['#2ecc71' if p in ['rdf', 'rdfs', 'owl', 'skos', 'xsd'] else '#3498db' 
          for p in top_ns['prefix']]

bars = ax.barh(range(len(top_ns)), top_ns['dataset_count'], color=colors, edgecolor='black')
ax.set_yticks(range(len(top_ns)))
ax.set_yticklabels(top_ns['prefix'])
ax.set_xlabel('Number of Datasets Using Namespace')
ax.set_title('Top 20 Most Used Namespaces Across LOD Cloud')
ax.invert_yaxis()

# Add percentage labels
for i, (count, pct) in enumerate(zip(top_ns['dataset_count'], top_ns['percent'])):
    ax.text(count + 0.5, i, f'{pct:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "namespace_usage.png", dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / "namespace_usage.pdf", bbox_inches='tight')
plt.show()

## 4. Cross-Dataset Connectivity

In [ ]:
# Compute pairwise Jaccard similarities
names = sorted(schemas.keys())
n = len(names)

# Initialize matrices
class_similarity = np.zeros((n, n))
property_similarity = np.zeros((n, n))

print(f"Computing {n*(n-1)//2} pairwise similarities...")

overlaps = []
for i, n1 in enumerate(names):
    for j, n2 in enumerate(names):
        if i >= j:
            continue
            
        class_sim = jaccard_similarity(
            schemas[n1]["classes"],
            schemas[n2]["classes"],
        )
        prop_sim = jaccard_similarity(
            schemas[n1]["properties"],
            schemas[n2]["properties"],
        )
        
        class_similarity[i, j] = class_sim
        class_similarity[j, i] = class_sim
        property_similarity[i, j] = prop_sim
        property_similarity[j, i] = prop_sim
        
        if class_sim > 0 or prop_sim > 0:
            overlaps.append({
                "source": n1,
                "target": n2,
                "class_jaccard": class_sim,
                "property_jaccard": prop_sim,
                "shared_classes": len(schemas[n1]["classes"] & schemas[n2]["classes"]),
                "shared_properties": len(schemas[n1]["properties"] & schemas[n2]["properties"]),
            })

df_overlaps = pd.DataFrame(overlaps).sort_values("class_jaccard", ascending=False)

print(f"Found {len(df_overlaps)} pairs with non-zero overlap")
print(f"Pairs with class overlap > 0.1: {len(df_overlaps[df_overlaps['class_jaccard'] > 0.1])}")
print(f"Pairs with property overlap > 0.1: {len(df_overlaps[df_overlaps['property_jaccard'] > 0.1])}")

In [ ]:
# Top overlapping pairs
print("\nTop 20 pairs by class Jaccard similarity:")
display(df_overlaps.head(20))

In [ ]:
# Figure: Connectivity heatmap (for top N datasets)
TOP_N = 30  # Show top N datasets by class count

top_datasets = df_schemas.head(TOP_N)["name"].tolist()
top_indices = [names.index(d) for d in top_datasets if d in names]

if len(top_indices) >= 5:
    sub_matrix = class_similarity[np.ix_(top_indices, top_indices)]
    sub_names = [names[i] for i in top_indices]

    fig, ax = plt.subplots(figsize=(14, 12))
    
    # Mask diagonal
    mask = np.eye(len(sub_matrix), dtype=bool)
    
    sns.heatmap(
        sub_matrix,
        mask=mask,
        xticklabels=sub_names,
        yticklabels=sub_names,
        cmap='YlOrRd',
        vmin=0,
        vmax=0.5,
        ax=ax,
        square=True,
        cbar_kws={'label': 'Jaccard Similarity (Classes)'}
    )
    ax.set_title(f'Class Overlap Heatmap (Top {TOP_N} Datasets)')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "connectivity_heatmap.png", dpi=300, bbox_inches='tight')
    plt.savefig(FIGURES_DIR / "connectivity_heatmap.pdf", bbox_inches='tight')
    plt.show()
else:
    print("Not enough data for heatmap")

In [ ]:
# Figure: Connectivity network graph
G = nx.Graph()

# Add nodes (datasets)
for name in schemas.keys():
    G.add_node(name, size=len(schemas[name]["classes"]))

# Add edges (overlaps above threshold)
THRESHOLD = 0.05
for _, row in df_overlaps.iterrows():
    if row["class_jaccard"] > THRESHOLD:
        G.add_edge(
            row["source"],
            row["target"],
            weight=row["class_jaccard"],
        )

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Connected components
components = list(nx.connected_components(G))
print(f"Connected components: {len(components)}")
print(f"Largest component: {len(max(components, key=len))} nodes")

# Find isolated nodes
isolated = [n for n in G.nodes() if G.degree(n) == 0]
print(f"Isolated nodes: {len(isolated)}")

In [ ]:
# Draw network (largest component only)
if G.number_of_edges() > 0:
    largest_cc = max(components, key=len)
    subgraph = G.subgraph(largest_cc)
    
    fig, ax = plt.subplots(figsize=(16, 12))
    
    # Layout
    pos = nx.spring_layout(subgraph, k=2, iterations=50, seed=42)
    
    # Node sizes based on class count
    sizes = [min(3000, 50 + len(schemas.get(n, {}).get("classes", [])) * 2) for n in subgraph.nodes()]
    
    # Edge widths based on similarity
    weights = [subgraph[u][v]["weight"] * 5 for u, v in subgraph.edges()]
    
    # Draw
    nx.draw_networkx_nodes(subgraph, pos, node_size=sizes, node_color='lightblue', alpha=0.8, ax=ax)
    nx.draw_networkx_edges(subgraph, pos, width=weights, alpha=0.3, ax=ax)
    nx.draw_networkx_labels(subgraph, pos, font_size=7, ax=ax)
    
    ax.set_title(f"LOD Cloud Connectivity (Jaccard > {THRESHOLD})")
    ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "connectivity_network.png", dpi=300, bbox_inches='tight')
    plt.savefig(FIGURES_DIR / "connectivity_network.pdf", bbox_inches='tight')
    plt.show()
else:
    print("No edges above threshold for network visualization")

## 5. Instance-Level Mappings

In [ ]:
# Analyze instance mappings
print("=" * 60)
print("INSTANCE MAPPING SUMMARY")
print("=" * 60)

total_edges = 0
mapping_stats = []

for source_type, files in mappings.items():
    type_edges = sum(f["edge_count"] for f in files)
    total_edges += type_edges
    
    mapping_stats.append({
        "source": source_type,
        "files": len(files),
        "edges": type_edges,
    })
    
    if files:
        print(f"{source_type}:")
        print(f"  Files: {len(files)}")
        print(f"  Total edges: {type_edges:,}")

print(f"\nTotal mapping edges: {total_edges:,}")

df_mappings = pd.DataFrame(mapping_stats)
display(df_mappings)

In [ ]:
# Figure: Mapping source distribution
if not df_mappings.empty and df_mappings["edges"].sum() > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # File counts
    df_non_zero = df_mappings[df_mappings["files"] > 0]
    if not df_non_zero.empty:
        axes[0].bar(df_non_zero["source"], df_non_zero["files"], color='steelblue', edgecolor='black')
        axes[0].set_xlabel("Mapping Source")
        axes[0].set_ylabel("Number of Files")
        axes[0].set_title("Mapping Files by Source")
        axes[0].tick_params(axis='x', rotation=45)
    
    # Edge counts
    df_edges = df_mappings[df_mappings["edges"] > 0]
    if not df_edges.empty:
        axes[1].bar(df_edges["source"], df_edges["edges"], color='coral', edgecolor='black')
        axes[1].set_xlabel("Mapping Source")
        axes[1].set_ylabel("Number of Edges")
        axes[1].set_title("Mapping Edges by Source")
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].ticklabel_format(style='scientific', axis='y', scilimits=(0, 0))
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "mapping_distribution.png", dpi=300, bbox_inches='tight')
    plt.savefig(FIGURES_DIR / "mapping_distribution.pdf", bbox_inches='tight')
    plt.show()
else:
    print("No mapping data available for visualization")

## 6. Paper Statistics Summary

In [ ]:
# Generate comprehensive statistics for the paper
paper_stats = {
    "overview": {
        "total_schemas": len(schemas),
        "total_classes": sum(len(s["classes"]) for s in schemas.values()),
        "total_properties": sum(len(s["properties"]) for s in schemas.values()),
        "total_patterns": sum(len(s["patterns"]) for s in schemas.values()),
        "unique_namespaces": len(namespace_usage),
    },
    "connectivity": {
        "overlapping_pairs": len(df_overlaps),
        "pairs_class_overlap_gt_01": len(df_overlaps[df_overlaps["class_jaccard"] > 0.1]),
        "pairs_class_overlap_gt_05": len(df_overlaps[df_overlaps["class_jaccard"] > 0.5]),
        "connected_components": len(components) if 'components' in dir() else 0,
        "isolated_datasets": len(isolated) if 'isolated' in dir() else 0,
    },
    "mappings": {
        "total_files": sum(len(v) for v in mappings.values()),
        "total_edges": total_edges,
        "by_source": {k: sum(f["edge_count"] for f in v) for k, v in mappings.items()},
    },
    "top_namespaces": df_ns.head(10).to_dict(orient="records") if not df_ns.empty else [],
}

# Save statistics
stats_path = OUTPUT_DIR / "paper_statistics.json"
stats_path.write_text(json.dumps(paper_stats, indent=2, default=str), encoding="utf-8")
print(f"Saved paper statistics to {stats_path}")

# Display
print("\n" + "=" * 60)
print("PAPER STATISTICS SUMMARY")
print("=" * 60)

display(Markdown(f"""
### Overview
- **Total schemas analyzed:** {paper_stats['overview']['total_schemas']}
- **Total unique classes:** {paper_stats['overview']['total_classes']:,}
- **Total unique properties:** {paper_stats['overview']['total_properties']:,}
- **Total patterns discovered:** {paper_stats['overview']['total_patterns']:,}
- **Unique namespaces:** {paper_stats['overview']['unique_namespaces']}

### Connectivity
- **Dataset pairs with overlap:** {paper_stats['connectivity']['overlapping_pairs']}
- **Pairs with Jaccard > 0.1:** {paper_stats['connectivity']['pairs_class_overlap_gt_01']}
- **Connected components:** {paper_stats['connectivity']['connected_components']}
- **Isolated datasets:** {paper_stats['connectivity']['isolated_datasets']}

### Mappings
- **Total mapping files:** {paper_stats['mappings']['total_files']}
- **Total mapping edges:** {paper_stats['mappings']['total_edges']:,}
"""))

In [ ]:
# List generated figures
print("\nGenerated figures:")
for f in sorted(FIGURES_DIR.glob("*")):
    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")